## CE13_Tinea_Vs_Candidiasis_Classifier

# Aim: To deploy a trained artificial intelligence (AI) model as an interactive web application using Streamlit and manage the deployment workflow using Git and GitHub for version control, collaboration, and cloud-based engineering AI solutions.

# Step 1: Install the libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Step 2: Dataset Paths

In [10]:
train_path = "dataset/train"
val_path = "dataset/valid"

# STEP 3 – Data Augmentation

In [11]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze the pretrained layers
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# STEP 4 – Load Dataset

In [12]:

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(
    rescale=1./255
)
train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(224,224),
    batch_size=32,
    class_mode="binary"
)

val_data = val_gen.flow_from_directory(
    val_path,
    target_size=(224,224),
    batch_size=32,
    class_mode="binary",
    shuffle=False
)

Found 4653 images belonging to 2 classes.
Found 1189 images belonging to 2 classes.


# STEP 5 – Verify Dataset

In [16]:
print(train_data.class_indices)
print("Training Images:", train_data.samples)
print("Validation Images:", val_data.samples)

{'Bacterial_spot': 0, 'Target_Spot': 1}
Training Images: 4653
Validation Images: 1189


# STEP 6 – Build MobileNetV2

In [17]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# STEP 7 – Compile

In [18]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# STEP 8 – Callbacks

In [19]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model_check = ModelCheckpoint(
    "models/tomato_classifier.keras",
    monitor="val_loss",
    save_best_only=True
)

# STEP 9 – Train the model

In [20]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop, model_check]
)

Epoch 1/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 2422s 15s/step - accuracy: 0.9409 - loss: 0.1548 - val_accuracy: 0.9849 - val_loss: 0.0544
Epoch 2/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 1162s 8s/step - accuracy: 0.9794 - loss: 0.0618 - val_accuracy: 0.9891 - val_loss: 0.0357
Epoch 3/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 795s 5s/step - accuracy: 0.9843 - loss: 0.0407 - val_accuracy: 0.9899 - val_loss: 0.0311
Epoch 4/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 442s 3s/step - accuracy: 0.9886 - loss: 0.0332 - val_accuracy: 0.9941 - val_loss: 0.0214
Epoch 5/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 273s 2s/step - accuracy: 0.9901 - loss: 0.0270 - val_accuracy: 0.9924 - val_loss: 0.0203
Epoch 6/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 267s 2s/step - accuracy: 0.9888 - loss: 0.0316 - val_accuracy: 0.9941 - val_loss: 0.0186
Epoch 7/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 286s 2s/step - accuracy: 0.9916 - loss: 0.0256 - val_accuracy: 0.9924 - val_loss: 0.0204
Epoch 8/20
146/146 ━━━━━━━━━━━━━━━━━━━━ 267s 2s/step - accuracy: 0.9923 - loss: 0.0226 - val_a

# STEP 10 – Save Model

In [21]:
model.save("models/tomato_classifier.keras")

# STEP 11 – Evaluate

In [22]:
loss, accuracy = model.evaluate(val_data)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)

38/38 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.9958 - loss: 0.0111
Validation Loss: 0.011124571785330772
Validation Accuracy: 0.9957947731018066


In [23]:
print(train_data.class_indices)

{'Bacterial_spot': 0, 'Target_Spot': 1}
